In [1]:
!pip install -q transformers datasets accelerate sentencepiece

In [18]:
import pandas as pd
import numpy as np
import torch
import random

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

In [19]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected!")

GPU available: True
GPU: Tesla T4


In [20]:
train_df = pd.read_csv("train_1.csv")
test_df = pd.read_csv("test_1.csv")

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

train_df.head()

Training shape: (4494, 3)
Test shape: (1284, 3)

Training labels:
label
real    2352
fake    2142
Name: count, dtype: int64

Test labels:
label
real    672
fake    612
Name: count, dtype: int64


,id,tweet,label
0,477,The novel coronavirus can be cured by one bowl...,fake
1,2434,At the peak in New York on April 15 that state...,real
2,4689,Video shows a huge crowd gathered in an Americ...,fake
3,5106,(2/4) Containment zones that refer to a specif...,real
4,2635,Australian Scientists referred to coronavirus ...,fake


In [21]:
# Remove rows with missing text or labels
train_df = train_df.dropna(subset=["tweet", "label"]).copy()
test_df = test_df.dropna(subset=["tweet", "label"]).copy()

# Make sure tweets are strings
train_df["tweet"] = train_df["tweet"].astype(str)
test_df["tweet"] = test_df["tweet"].astype(str)

# Standardize labels
train_df["label"] = train_df["label"].astype(str).str.lower().str.strip()
test_df["label"] = test_df["label"].astype(str).str.lower().str.strip()

# Convert labels to numbers
# fake = 0
# real = 1
label_map = {
    "fake": 0,
    "real": 1
}

train_df["labels"] = train_df["label"].map(label_map)
test_df["labels"] = test_df["label"].map(label_map)

print("Training label counts:")
print(train_df["labels"].value_counts())

print("\nTest label counts:")
print(test_df["labels"].value_counts())

print("\nMissing training labels:", train_df["labels"].isna().sum())
print("Missing test labels:", test_df["labels"].isna().sum())

Training label counts:
labels
1    2352
0    2142
Name: count, dtype: int64

Test label counts:
labels
1    672
0    612
Name: count, dtype: int64

Missing training labels: 0
Missing test labels: 0


In [22]:
train_dataset = Dataset.from_pandas(
    train_df[["tweet", "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["tweet", "labels"]],
    preserve_index=False
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['tweet', 'labels'],
    num_rows: 4494
})
Dataset({
    features: ['tweet', 'labels'],
    num_rows: 1284
})


In [23]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded!")

Tokenizer loaded!


In [24]:
def tokenize_function(examples):
    return tokenizer(
        examples["tweet"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization complete!")
print(train_tokenized)
print(test_tokenized)

Map:   0%|          | 0/4494 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Tokenization complete!
Dataset({
    features: ['tweet', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4494
})
Dataset({
    features: ['tweet', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1284
})


In [25]:
id2label = {
    0: "fake",
    1: "real"
}

label2id = {
    "fake": 0,
    "real": 1
}

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32
)

print("DeBERTa loaded!")
print("Model dtype:", next(model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

DeBERTa loaded!
Model dtype: torch.float32


In [27]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [28]:
training_args = TrainingArguments(
    output_dir="./deberta_results",

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=2,

    # IMPORTANT: keep both OFF
    fp16=False,
    bf16=False,

    report_to="none",

    seed=SEED
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [30]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model dtype:", next(model.parameters()).dtype)

print("\nTraining examples:", len(train_tokenized))
print("Test examples:", len(test_tokenized))

GPU available: True
GPU: Tesla T4
Model dtype: torch.float32

Training examples: 4494
Test examples: 1284


In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.116415,0.135759,0.969626,0.969643,0.969626,0.969629
2,0.072131,0.129940,0.974299,0.974681,0.974299,0.974276
3,0.019912,0.131412,0.975857,0.975901,0.975857,0.975850


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1686, training_loss=0.12291914143313434, metrics={'train_runtime': 813.1597, 'train_samples_per_second': 16.58, 'train_steps_per_second': 2.073, 'total_flos': 886831716105216.0, 'train_loss': 0.12291914143313434, 'epoch': 3.0})

In [32]:
results = trainer.evaluate(test_tokenized)

print("\nDEBERTA TEST RESULTS")
print("=" * 40)

print(f"Accuracy:  {results['eval_accuracy'] * 100:.2f}%")
print(f"Precision: {results['eval_precision'] * 100:.2f}%")
print(f"Recall:    {results['eval_recall'] * 100:.2f}%")
print(f"F1 Score:  {results['eval_f1'] * 100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.019912,0.131412,3,0.975857,0.975901,0.975857,0.975850



DEBERTA TEST RESULTS
Accuracy:  97.59%
Precision: 97.59%
Recall:    97.59%
F1 Score:  97.58%


In [33]:
predictions_output = trainer.predict(test_tokenized)

predictions = np.argmax(
    predictions_output.predictions,
    axis=-1
)

true_labels = predictions_output.label_ids

print("DEBERTA CLASSIFICATION REPORT")
print("=" * 50)

print(
    classification_report(
        true_labels,
        predictions,
        target_names=["fake", "real"],
        digits=4
    )
)

DEBERTA CLASSIFICATION REPORT
              precision    recall  f1-score   support

        fake     0.9802    0.9690    0.9745       612
        real     0.9720    0.9821    0.9771       672

    accuracy                         0.9759      1284
   macro avg     0.9761    0.9755    0.9758      1284
weighted avg     0.9759    0.9759    0.9758      1284



In [34]:
cm = confusion_matrix(true_labels, predictions)

print("DEBERTA CONFUSION MATRIX")
print("=" * 40)

print(cm)

print("\nInterpretation:")
print(f"Fake correctly classified: {cm[0][0]}")
print(f"Fake incorrectly classified as real: {cm[0][1]}")
print(f"Real incorrectly classified as fake: {cm[1][0]}")
print(f"Real correctly classified: {cm[1][1]}")

DEBERTA CONFUSION MATRIX
[[593  19]
 [ 12 660]]

Interpretation:
Fake correctly classified: 593
Fake incorrectly classified as real: 19
Real incorrectly classified as fake: 12
Real correctly classified: 660


In [35]:
total_errors = np.sum(predictions != true_labels)
correct_predictions = len(true_labels) - total_errors

print("Total test examples:", len(true_labels))
print("Correct predictions:", correct_predictions)
print("Incorrect predictions:", total_errors)
print(f"Error rate: {(total_errors / len(true_labels)) * 100:.2f}%")

Total test examples: 1284
Correct predictions: 1253
Incorrect predictions: 31
Error rate: 2.41%


In [36]:
error_indices = np.where(predictions != true_labels)[0]

errors_df = test_df.iloc[error_indices].copy()

errors_df["true_label"] = [
    id2label[x] for x in true_labels[error_indices]
]

errors_df["predicted_label"] = [
    id2label[x] for x in predictions[error_indices]
]

print("Number of misclassified tweets:", len(errors_df))

errors_df[
    ["tweet", "true_label", "predicted_label"]
].head(20)

Number of misclassified tweets: 31


,tweet,true_label,predicted_label
17,Testing doesn’t create the virus. Even without...,fake,real
146,Another reason is the casual attitude of peopl...,real,fake
187,U.S. sees 35K new COVID cases; national total ...,fake,real
198,The information circulating on social media th...,real,fake
204,#eVIN is presently being implemented by the Ce...,real,fake
248,"In AURANGABA, Covid antibody testing in Privat...",fake,real
250,There's now 3 different #coronavirus vaccines ...,fake,real
272,Researchers warned of a ticking coronavirus ti...,real,fake
353,"8th September #coronavirus update:\n\n🔹️89,852...",fake,real
432,Every worker on the front lines of the fight a...,fake,real


In [37]:
comparison = pd.DataFrame({
    "Model": [
        "RoBERTa",
        "DistilBERT",
        "Stacking Ensemble",
        "CNN-LSTM",
        "DeBERTa"
    ],

    "Test Accuracy (%)": [
        97.51,
        96.65,
        93.85,
        91.51,
        results["eval_accuracy"] * 100
    ]
})

comparison = comparison.sort_values(
    "Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

comparison

,Model,Test Accuracy (%)
0,DeBERTa,97.58567
1,RoBERTa,97.51000
2,DistilBERT,96.65000
3,Stacking Ensemble,93.85000
4,CNN-LSTM,91.51000
